In [ ]:
# ================= CLEAN OLD PACKAGES =================
!pip uninstall -y numpy opencv-python opencv-python-headless ultralytics easyocr lapx torch torchvision torchaudio

# ================= INSTALL EXACT COMPATIBLE STACK =================
!pip install --upgrade pip
!pip install numpy==1.26.4
!pip install torch==2.2.2 torchvision==0.17.2 torchaudio==2.2.2
!pip install ultralytics==8.2.103 easyocr==1.7.1 opencv-python-headless==4.10.0.84 lapx==0.5.5 pyngrok==7.1.6 flask==3.0.3 requests==2.31.0

In [ ]:
import numpy as np
import torch

# Print installed versions to confirm the correct packages are active.
print("NumPy:", np.__version__)
print("Torch:", torch.__version__)

# Verify that PyTorch can accept NumPy arrays directly.
# torch.from_numpy() breaks on NumPy 2.x due to a dtype API change;
# this test catches that issue early.
x = np.zeros((2, 2), dtype=np.float32)
try:
    t = torch.from_numpy(x)
    print(" Torchâ€“NumPy bridge OK")
except TypeError as e:
    print(f" Torch-Numpy bridge issue: {e}")
    # Fallback: convert the NumPy array to a Python list first,
    # then wrap it in a torch.tensor â€” avoids the dtype incompatibility.
    t = torch.tensor(x.tolist(), dtype=torch.float32)
    print(" Using torch.tensor with tolist() bypass")

print(t)


NumPy: 1.26.4
Torch: 2.2.2+cu121
âœ… Torchâ€“NumPy bridge OK
tensor([[0., 0.],
        [0., 0.]])


In [ ]:
# Kill any leftover ngrok processes from a previous run to avoid port conflicts.
from pyngrok import ngrok
ngrok.kill()

import os
from pyngrok import ngrok
import subprocess, threading, time

# Define the camera feeds available to the system.
# ============================================================
#  VIDEO PATHS  â€”  Change these whenever you want
# ============================================================
CAMERA_FEEDS = {
    'cam1': {
        'path':     '/kaggle/input/datasets/shiranofficial/roadsurveillance-testdata/track2.mp4',
        'location': 'Main Street Intersection'
    },
    'cam2': {
        'path':     '/kaggle/input/datasets/shiranofficial/roadsurveillance-testdata/cam1.mp4',
        'location': 'Highway Exit 42'
    },
    'cam3': {
        'path':     '/kaggle/input/datasets/shiranofficial/roadsurveillance-testdata/track3.mp4',
        'location': 'Downtown Plaza'
    },
}

# Publish each camera's path and location as environment variables so the Flask
# server process can access them without needing them passed as arguments.
for cam_id, info in CAMERA_FEEDS.items():
    os.environ[f"{cam_id.upper()}_PATH"]     = info['path']
    os.environ[f"{cam_id.upper()}_LOCATION"] = info['location']

# Store the total number of registered cameras so the server knows how many to expect.
os.environ["CAMERA_COUNT"] = str(len(CAMERA_FEEDS))

print(" Camera feeds registered:")
for cam_id, info in CAMERA_FEEDS.items():
    print(f"   {cam_id} â†’ {info['path']}  ({info['location']})")

# Authenticate with ngrok using the project's auth token, then open a public
# HTTPS tunnel pointing to the Flask server on port 5001.
# The printed URL is what the Django backend should use to reach this Flask API.
ngrok.set_auth_token("399CjUS3EVgtcwrBJGqltyWI7NB_6S7WRWwVGppZ8ZgmktVo7")

tunnel = ngrok.connect(5001)
print(f"\n YOUR NGROK URL IS: {tunnel.public_url}")
print(f"   â†’ Use this URL to access the Flask APIs\n")

 Camera feeds registered:
   cam1 â†’ /kaggle/input/datasets/shiranofficial/roadsurveillance-testdata/track2.mp4  (Main Street Intersection)
   cam2 â†’ /kaggle/input/datasets/shiranofficial/roadsurveillance-testdata/cam1.mp4  (Highway Exit 42)
   cam3 â†’ /kaggle/input/datasets/shiranofficial/roadsurveillance-testdata/track3.mp4  (Downtown Plaza)

 YOUR NGROK URL IS: https://bausond-martially-cherly.ngrok-free.dev
   â†’ Use this URL to access the Flask APIs



In [ ]:
import torch
import cv2
import numpy as np
import easyocr
import os
import base64
import requests
from ultralytics import YOLO
from collections import Counter, deque
from difflib import SequenceMatcher
import threading
from queue import Queue
from datetime import datetime
from flask import Flask, request, jsonify

# Base URL of the Django backend that this Flask service communicates with.
# All incident reports and search results are posted to these endpoints.
DJANGO_BASE_URL = "https://nonclimbable-muddledly-jami.ngrok-free.dev"
CREATE_INCIDENT_API = f"{DJANGO_BASE_URL}/api/incident/create/"
SEARCH_RESULT_API = f"{DJANGO_BASE_URL}/api/search/result/"

# Paths to the YOLO model weights.
# custom_model: fine-tuned model for detecting accidents and fights.
# tracking_model: general YOLOv8 medium model used for vehicle detection and plate scanning.
CUSTOM_MODEL_PATH = '/kaggle/input/models/shiranofficial/roadmodel/other/default/1/best.pt'
TRACKING_MODEL_PATH = 'yolov8m.pt'

# Tunable thresholds that control detection sensitivity and plate validation.
CONFIDENCE_THRESHOLD = 0.60     # Minimum YOLO confidence score to accept a detection.
PLATE_MATCH_CONFIDENCE = 0.8    # Minimum string-similarity ratio to accept a plate match.
PRE_IMPACT_SECONDS = 4          # Seconds before the incident frame to begin plate extraction.
BLUR_THRESHOLD = 80.0           # Laplacian variance below this value means the crop is too blurry for OCR.
IMPACT_ZONE_RADIUS = 120        # Pixel radius around impact centers; only cars inside are candidates.
DETECTION_WINDOW_SIZE = 30      # Rolling window (frames) used for smoothing incident detection scores.
ACTIVATION_THRESHOLD = 15       # Number of positive detections within the window needed to confirm an incident.

# Select GPU if available; otherwise fall back to CPU.
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"  Device: {device.upper()}")

# Load the custom surveillance model for accident/fight classification.
print(" Loading custom surveillance model...")
custom_model = YOLO(CUSTOM_MODEL_PATH).to(device)

# Load the general tracking model used for vehicle detection and plate scanning.
print(" Loading tracking model (for plate/camera work)...")
tracking_model = YOLO(TRACKING_MODEL_PATH).to(device)

# Load the general tracking model used for vehicle detection and plate scanning.
reader = easyocr.Reader(['en'], gpu=(device == 'cuda'))

# Create output directories for storing forensic reports and per-camera tracking evidence.
os.makedirs('forensics_reports', exist_ok=True)
os.makedirs('tracking_evidence', exist_ok=True)

# Thread-safe queue that camera scanning threads push their results into.
# A lock serialises YOLO model loads so two threads don't compete for GPU memory.
results_queue = Queue()
thread_lock = threading.Lock()

# ============================================================
# HELPER FUNCTIONS
# ============================================================
def image_to_base64(img):
    """Encode a BGR OpenCV image as a JPEG base64 string for JSON serialisation."""
    _, buffer = cv2.imencode('.jpg', img)
    return base64.b64encode(buffer).decode()

def forensic_upscale(img):
    """Upscale a small plate crop 3x and apply CLAHE to improve OCR accuracy.
    Returns a contrast-enhanced greyscale image, or None if the input is empty."""
    if img.size == 0:
        return None
    img = cv2.resize(img, None, fx=3.0, fy=3.0, interpolation=cv2.INTER_LANCZOS4)
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    clahe = cv2.createCLAHE(clipLimit=5.0, tileGridSize=(8, 8))
    return clahe.apply(gray)

def is_blurry(gray, threshold=BLUR_THRESHOLD):
    """Return True if the greyscale image is too blurry to read.
    Uses the variance of the Laplacian as a focus measure."""
    return cv2.Laplacian(gray, cv2.CV_64F).var() < threshold

def clean_text(text):
    """Strip non-alphanumeric characters from OCR output and convert to uppercase."""
    return "".join(e for e in text if e.isalnum()).upper()

def similar(a, b):
    """Return a 0â€“1 similarity ratio between two strings using SequenceMatcher."""
    return SequenceMatcher(None, a, b).ratio()

def get_video_fps(path):
    """Read the frame rate of a video file. Falls back to 30 FPS if unavailable."""
    cap = cv2.VideoCapture(path)
    fps = cap.get(cv2.CAP_PROP_FPS)
    cap.release()
    return fps if fps > 0 else 30

def frame_to_timestamp(frame, fps):
    """Convert a frame index to a MM:SS string given the video's frame rate."""
    sec = frame / fps
    return f"{int(sec // 60):02d}:{int(sec % 60):02d}"

def center(box):
    """Return the centroid [x, y] of an axis-aligned bounding box [x1,y1,x2,y2]."""
    return np.array([(box[0] + box[2]) / 2, (box[1] + box[3]) / 2])

def distance(p1, p2):
    """Euclidean distance between two 2-D points."""
    return np.linalg.norm(np.array(p1) - np.array(p2))

def iou(box1, box2):
    """Intersection-over-Union of two bounding boxes.
    Returns a value in [0, 1]; higher means more overlap."""
    x1 = max(box1[0], box2[0])
    y1 = max(box1[1], box2[1])
    x2 = min(box1[2], box2[2])
    y2 = min(box1[3], box2[3])
    inter = max(0, x2 - x1) * max(0, y2 - y1)
    area1 = (box1[2] - box1[0]) * (box1[3] - box1[1])
    area2 = (box2[2] - box2[0]) * (box2[3] - box2[1])
    union = area1 + area2 - inter
    return inter / union if union > 0 else 0

def extract_impact_centers(frame):
    """Detect all cars in a frame and return their centroids.
    Used to identify the spatial zone of an accident for targeted plate extraction."""
    results = tracking_model.predict(frame, conf=0.4, verbose=False)
    centers = []
    if results[0].boxes is not None:
        boxes = results[0].boxes.xyxy.cpu().numpy().astype(int)
        clss = results[0].boxes.cls.cpu().numpy().astype(int)
        for box, cls in zip(boxes, clss):
            if results[0].names[cls] == 'car':
                centers.append(center(box))
    return centers

Disabling PyTorch because PyTorch >= 2.4 is required but found 2.2.2
PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


  Device: CUDA
 Loading custom surveillance model...
 Loading tracking model (for plate/camera work)...


In [ ]:
# ============================================================
# CAMERA SCANNING WORKER
# ============================================================
def scan_camera_feed(cam_id, cam_info, target_plate):
    """Thread worker that scans a single camera's video for a target license plate.

    For each frame the function:
      1. Runs YOLO tracking to locate cars.
      2. Crops the lower half of each car bounding box (where the plate is).
      3. Upscales and enhances the crop, then runs EasyOCR.
      4. Compares the OCR result to the target plate using exact substring
         match or fuzzy similarity.
    On the first match the function saves a snapshot and exits early.
    The final result (match or None) is placed on the shared results_queue.
    """
    print(f"  [{cam_id}] Thread started - Scanning {cam_info['location']}...")

    clean_target = clean_text(target_plate)
    cap = cv2.VideoCapture(cam_info['path'])

    vehicle_found = False
    detection_time = None
    best_match_snapshot = None
    found_plate_text = ""

    # Load a dedicated YOLO instance and OCR reader per thread.
    # The thread_lock ensures GPU memory allocation is serialised.
    with thread_lock:
        local_model = YOLO(TRACKING_MODEL_PATH).to(device)
    local_reader = easyocr.Reader(['en'], gpu=(device == 'cuda'))

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break

        # Run YOLO with persistent tracking so vehicle IDs are stable across frames.
        results = local_model.track(frame, persist=True, verbose=False)

        if results[0].boxes.id is not None:
            boxes = results[0].boxes.xyxy.cpu().numpy().astype(int)
            ids = results[0].boxes.id.cpu().numpy().astype(int)
            clss = results[0].boxes.cls.cpu().numpy().astype(int)

            for box, id, cls in zip(boxes, ids, clss):
                if results[0].names[cls] == 'car':
                    x1, y1, x2, y2 = box
                    h = y2 - y1
                    # Crop the bottom 50 % of the car box â€” that's where the plate sits.
                    bumper_crop = frame[int(y1 + h * 0.50):y2, x1:x2]

                    # Upscale and enhance for better OCR performance.
                    processed_plate = forensic_upscale(bumper_crop)

                    if processed_plate is not None:
                        # Run OCR restricted to alphanumeric characters only.
                        ocr_hits = local_reader.readtext(
                            processed_plate,
                            detail=0,
                            allowlist='0123456789ABCDEFGHIJKLMNOPQRSTUVWXYZ'
                        )

                        for text in ocr_hits:
                            detected_clean = clean_text(text)
                            # Accept the detection if the target plate appears as a
                            # substring of the OCR result, or if the two strings are
                            # sufficiently similar (fuzzy match).
                            if (clean_target in detected_clean) or (similar(clean_target, detected_clean) > PLATE_MATCH_CONFIDENCE):
                                if not vehicle_found:
                                    vehicle_found = True
                                    detection_time = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
                                    found_plate_text = text
                                    best_match_snapshot = frame.copy()
                                    # Annotate the snapshot with a bounding box and timestamp.
                                    cv2.rectangle(best_match_snapshot, (x1, y1), (x2, y2), (0, 255, 0), 4)
                                    cv2.putText(best_match_snapshot, f"FOUND: {detection_time}",
                                                (x1, y1 - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 0), 2)
                                    print(f"       [{cam_id}] MATCH FOUND at {detection_time}")
                                break
                        if vehicle_found:
                            break
            if vehicle_found:
                break
        if vehicle_found:
            break

    cap.release()

    # Build the result dictionary if the plate was found; otherwise result stays None.
    result = None
    if vehicle_found:
        result = {
            'camera': cam_id,
            'location': cam_info['location'],
            'timestamp': detection_time,
            'matched_text': found_plate_text,
            'snapshot': best_match_snapshot
        }

    # Push the result (match or no-match) onto the shared queue for the caller to collect.
    results_queue.put({
        'cam_id': cam_id,
        'plate': target_plate,
        'result': result
    })

    if not vehicle_found:
        print(f"       [{cam_id}] Scan complete - No match found")

In [ ]:
# ============================================================
# FLASK APP
# ============================================================
app = Flask(__name__)

@app.route('/setup_camera_feeds', methods=['POST'])
def setup_camera_feeds():
    """Update the global camera registry at runtime without restarting the server.

    Expects JSON: { cam_id: { path: str, location: str }, ... }
    Publishes each entry to environment variables and returns the camera count.
    """
    data = request.json
    global CAMERA_FEEDS
    CAMERA_FEEDS = data
    # Publish updated paths as environment variables so other components can read them.
    for cam_id, info in CAMERA_FEEDS.items():
        os.environ[f"{cam_id.upper()}_PATH"] = info['path']
        os.environ[f"{cam_id.upper()}_LOCATION"] = info['location']
    os.environ["CAMERA_COUNT"] = str(len(CAMERA_FEEDS))
    return jsonify({"status": "success", "cameras_registered": len(CAMERA_FEEDS)})

@app.route('/detect_incident', methods=['POST'])
def detect_incident():
    """Synchronously scan a video for accident or fight incidents.

    Runs the custom YOLO model frame-by-frame and accumulates detections
    in two rolling deques (one per incident type). An incident is confirmed
    when the rolling sum crosses ACTIVATION_THRESHOLD within the window.
    Returns incident type, frame index, timestamp, base64 snapshot, and
    impact zone centroids. Returns {'status': 'no_incident'} if nothing found.
    """
    data = request.json
    video_path = data.get('video_path', '/kaggle/input/datasets/shiranofficial/roadsurveillance-testdata/accident.mp4')
    confidence_threshold = data.get('confidence_threshold', CONFIDENCE_THRESHOLD)
    window_size = data.get('window_size', DETECTION_WINDOW_SIZE)
    activation_threshold = data.get('activation_threshold', ACTIVATION_THRESHOLD)

    cap = cv2.VideoCapture(video_path)
    fps = get_video_fps(video_path)

    incident_snapshot = None
    incident_frame = 0
    impact_centers = []
    incident_type = None

    # Sliding-window accumulators: append 1 on positive detection, 0 otherwise.
    recent_accidents = deque(maxlen=window_size)
    recent_fights = deque(maxlen=window_size)

    frame_idx = 0

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break
        frame_idx += 1

        # Run the custom model and collect predicted class names for this frame.
        results = custom_model.predict(frame, conf=confidence_threshold, verbose=False)
        frame_classes = [custom_model.names[int(box.cls[0].item())] for box in results[0].boxes]

        is_acc = 'accident' in frame_classes
        is_fgt = 'fight' in frame_classes

        recent_accidents.append(1 if is_acc else 0)
        recent_fights.append(1 if is_fgt else 0)

        acc_score = sum(recent_accidents)
        fgt_score = sum(recent_fights)

        # Confirm an accident: save snapshot and locate impact centroids for plate extraction.
        if acc_score >= activation_threshold:
            incident_type = 'accident'
            incident_snapshot = frame.copy()
            incident_frame = frame_idx
            impact_centers = extract_impact_centers(frame)
            break
        elif fgt_score >= activation_threshold:
            incident_type = 'fight'
            incident_snapshot = frame.copy()
            incident_frame = frame_idx
            break

    cap.release()

    if incident_type:
        print({
            "incident_type": incident_type,
            "frame": incident_frame,
            "timestamp": frame_to_timestamp(incident_frame, fps),
            "snapshot_base64": image_to_base64(incident_snapshot),
            "impact_centers": impact_centers
        })
        return jsonify({
            "incident_type": incident_type,
            "frame": incident_frame,
            "timestamp": frame_to_timestamp(incident_frame, fps),
            "snapshot_base64": image_to_base64(incident_snapshot),
            "impact_centers": impact_centers
        })
    else:
        return jsonify({"status": "no_incident"})

@app.route('/extract_plates', methods=['POST'])
def extract_plates():
    """Extract license plates from the video segment just before an accident.

    Scans frames from (incident_frame - pre_impact_seconds * fps) to incident_frame.
    Only examines cars whose centre falls within IMPACT_ZONE_RADIUS of a known
    impact centre. Applies forensic upscaling, blur filtering, and OCR.

    Plate selection strategy:
      1. Prefer plates matching the 6-char 3-letter/3-digit format (most common).
      2. Fall back to prefix-frequency + character-position scoring.

    Returns the best plate(s) and a debug statistics dictionary.
    """
    data = request.json
    video_path = data.get('video_path')
    incident_frame = data.get('incident_frame')
    impact_centers = data.get('impact_centers', [])
    pre_impact_seconds = data.get('pre_impact_seconds', PRE_IMPACT_SECONDS)

    if not video_path or not incident_frame:
        return jsonify({"error": "video_path and incident_frame required"}), 400

    fps = get_video_fps(video_path)
    pre_impact_frames = int(fps * pre_impact_seconds)
    start_frame = max(0, incident_frame - pre_impact_frames)
    end_frame = incident_frame

    cap = cv2.VideoCapture(video_path)
    plate_ballot = []

    # Counters to help debug the plate extraction pipeline.
    debug_stats = {"frames_scanned": 0, "cars_seen": 0, "cars_near_impact": 0, "bumper_crops": 0, "ocr_runs": 0, "ocr_texts": 0}

    frame_idx = 0
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break
        frame_idx += 1
        if frame_idx < start_frame:
            continue
        if frame_idx > end_frame:
            break

        # Use persistent tracking so each vehicle keeps the same ID across frames.
        results = tracking_model.track(frame, persist=True, conf=0.4, verbose=False)

        if results[0].boxes.id is None:
            continue

        boxes = results[0].boxes.xyxy.cpu().numpy().astype(int)
        ids = results[0].boxes.id.cpu().numpy().astype(int)
        clss = results[0].boxes.cls.cpu().numpy().astype(int)

        debug_stats["frames_scanned"] += 1

        for box, id, cls in zip(boxes, ids, clss):
            if results[0].names[cls] == 'car':
                debug_stats["cars_seen"] += 1
                cx = (box[0] + box[2]) // 2
                cy = (box[1] + box[3]) // 2

                # Skip vehicles that are not near the collision zone.
                if impact_centers and not any(np.linalg.norm(np.array([cx, cy]) - np.array(ic)) < IMPACT_ZONE_RADIUS for ic in impact_centers):
                    continue

                debug_stats["cars_near_impact"] += 1

                x1, y1, x2, y2 = box
                h = y2 - y1
                # Crop the bottom half of the vehicle bounding box with a small margin.
                bumper_crop = frame[int(y1 + h * 0.50): y2 + 5, max(0, x1 - 5): min(frame.shape[1], x2 + 5)]
                debug_stats["bumper_crops"] += 1

                processed = forensic_upscale(bumper_crop)
                if processed is None:
                    continue
                # Reject blurry crops â€” OCR on motion-blurred plates is unreliable.
                if is_blurry(processed):
                    continue

                ocr_hits = reader.readtext(processed, detail=0, allowlist='0123456789ABCDEFGHIJKLMNOPQRSTUVWXYZ')
                debug_stats["ocr_runs"] += 1

                for text in ocr_hits:
                    detected_clean = clean_text(text)
                    # Remove a leading '0' or '1' that OCR sometimes misreads as a prefix.
                    if len(detected_clean) > 6 and detected_clean[0] in "01":
                        detected_clean = detected_clean[1:]
                    if len(detected_clean) >= 6:
                        plate_ballot.append(detected_clean)
                        debug_stats["ocr_texts"] += 1

    cap.release()

    # Select the best plate from all OCR readings.
    if plate_ballot:
        # Prefer plates that exactly match the expected 6-char format (3 letters + 3 digits).
        valid_plates = []
        for plate in plate_ballot:
            if len(plate) == 6:
                letters = sum(1 for c in plate if c.isalpha())
                digits = sum(1 for c in plate if c.isdigit())
                if letters == 3 and digits == 3:
                    valid_plates.append(plate)

        if valid_plates:
            # Pick the most frequently seen valid plate as the consensus reading.
            valid_counter = Counter(valid_plates)
            most_common = valid_counter.most_common(1)[0]
            detected_plates = [most_common[0]]
        else:
            # Fallback: group by the most common 3-character prefix, then score
            # candidates by letter count, occurrence frequency, and per-position
            # character frequency to pick the most credible partial reading.
            prefixes = [plate[:3] for plate in plate_ballot if len(plate) >= 3]
            if prefixes:
                prefix_counter = Counter(prefixes)
                most_common_prefix = prefix_counter.most_common(1)[0][0]
                prefix_plates = [p for p in plate_ballot if p.startswith(most_common_prefix)]
                unique_plates = list(set(prefix_plates))
                scored_plates = []
                for plate in unique_plates:
                    letter_count = sum(1 for c in plate if c.isalpha())
                    occurrence_count = prefix_plates.count(plate)
                    char_pos_freq = {}
                    for pos, char in enumerate(plate):
                        key = (pos, char)
                        char_pos_freq[key] = char_pos_freq.get(key, 0) + 1
                    freq_score = sum(char_pos_freq.get((pos, char), 0) for pos, char in enumerate(plate))
                    total_score = (letter_count * 10) + (occurrence_count * 5) + freq_score
                    scored_plates.append((plate, total_score))
                scored_plates.sort(key=lambda x: x[1], reverse=True)
                if scored_plates:
                    detected_plates = [scored_plates[0][0]]
                else:
                    detected_plates = []
            else:
                detected_plates = []
    else:
        detected_plates = []

    return jsonify({"detected_plates": detected_plates, "debug_stats": debug_stats})

@app.route('/track_vehicle', methods=['POST'])
def track_vehicle():
    """Search all registered camera feeds for a specific license plate (synchronous).

    Spawns one thread per camera, waits for all to complete, then returns
    detection records (camera, location, timestamp, snapshot path).
    Matched snapshots are saved to the tracking_evidence/ directory.
    """
    data = request.json
    plate = data.get('plate')
    camera_feeds = data.get('camera_feeds', CAMERA_FEEDS)

    if not plate:
        return jsonify({"error": "plate required"}), 400

    vehicle_routes = {plate: []}

    # Launch one scanning thread per camera for parallel processing.
    threads = []
    for cam_id, cam_info in camera_feeds.items():
        thread = threading.Thread(target=scan_camera_feed, args=(cam_id, cam_info, plate))
        threads.append(thread)
        thread.start()

    for thread in threads:
        thread.join()

    # Drain the results queue, record routes, and persist matched snapshots.
    while not results_queue.empty():
        result_data = results_queue.get()
        if result_data['plate'] == plate and result_data['result'] is not None:
            detection = result_data['result']
            vehicle_routes[plate].append(detection)
            save_path = f"tracking_evidence/{plate}_{result_data['cam_id']}.jpg"
            cv2.imwrite(save_path, detection['snapshot'])

    return jsonify({"plate": plate, "routes": vehicle_routes[plate]})

@app.route('/submit_incident', methods=['POST'])
def submit_incident():
    """Forward an assembled incident report to the Django backend.

    Accepts incident type, location, timestamp, a base64 snapshot, and
    a list of involved vehicles, then POSTs them to CREATE_INCIDENT_API.
    """
    data = request.json
    incident_type = data.get('incident_type')
    location = data.get('location', 'Main Feed')
    timestamp = data.get('timestamp', datetime.now().strftime("%Y-%m-%d %H:%M:%S"))
    snapshot_base64 = data.get('snapshot_base64')
    vehicles = data.get('vehicles', [])

    payload = {
        "incident_type": incident_type,
        "location": location,
        "timestamp": timestamp,
        "snapshot": snapshot_base64,
        "vehicles": vehicles
    }

    try:
        response = requests.post(CREATE_INCIDENT_API, json=payload, timeout=10)
        if response.status_code in (200, 201):
            return jsonify({"status": "success", "response": response.json()})
        else:
            return jsonify({"status": "error", "code": response.status_code, "response": response.text})
    except requests.exceptions.RequestException as e:
        return jsonify({"status": "error", "message": str(e)})

@app.route('/status', methods=['GET'])
def status():
    """Health-check endpoint. Returns server status, the active ngrok URL,
    the list of loaded model names, and the compute device in use."""
    return jsonify({
        "status": "running",
        "ngrok_url": tunnel.public_url if 'tunnel' in globals() else None,
        "models_loaded": ["custom_yolo", "tracking_yolo"],
        "device": device
    })


# Background worker: license-plate search across all cameras.
def search_and_callback(search_id, license_plate, callback_url):
    """Search every registered camera for a plate and POST results to Django.

    Steps:
      1. Spawn one scan_camera_feed thread per camera.
      2. Collect results from the shared queue once all threads finish.
      3. Convert numpy snapshots to base64 for JSON serialisation.
      4. Sort detections chronologically and POST to callback_url.
      5. On failure, send an empty fallback so Django is not left waiting.
    """
    try:
        print(f"\n  Starting search for plate: {license_plate}")

        # Launch parallel camera scans.
        threads = []
        for cam_id, cam_info in CAMERA_FEEDS.items():
            thread = threading.Thread(
                target=scan_camera_feed,
                args=(cam_id, cam_info, license_plate)
            )
            threads.append(thread)
            thread.start()

        for thread in threads:
            thread.join()

        # Collect all successful detections from the shared results queue.
        detections = []
        while not results_queue.empty():
            result_data = results_queue.get()
            if result_data.get('result') is not None:
                det = result_data['result']
                # Convert numpy snapshot arrays to base64 strings before JSON serialisation.
                if det.get('snapshot') is not None and not isinstance(det['snapshot'], str):
                    det['snapshot'] = image_to_base64(det['snapshot'])
                detections.append(det)

        # Sort chronologically so Django receives sightings in the order they occurred.
        detections.sort(key=lambda d: d.get('timestamp', ''))

        payload = {
            "search_id": search_id,
            "plate": license_plate,
            "confidence": 0.9,
            "detections": detections
        }

        print(f"  Posting search results to Django: {callback_url}")
        print(f"   search_id={search_id}, plate={license_plate}, detections={len(detections)}")

        response = requests.post(callback_url, json=payload, timeout=30)

        if response.status_code in (200, 201):
            print(f"  Django received results (HTTP {response.status_code})")
        else:
            print(f"  Django returned HTTP {response.status_code}: {response.text}")

    except Exception as e:
        print(f"  Error in search_and_callback: {e}")
        # Send an empty result so Django can mark the search as processed.
        try:
            fallback = {
                "search_id": search_id,
                "plate": license_plate,
                "confidence": 0.0,
                "detections": []
            }
            requests.post(callback_url, json=fallback, timeout=10)
            print("  Fallback empty-result callback sent.")
        except Exception as fe:
            print(f"  Fallback callback also failed: {fe}")


# ============================================================
# ENDPOINT: /search  (for vehicle tracking)
# ============================================================
@app.route('/search', methods=['POST'])
def search_endpoint():
    """Async endpoint for cross-camera vehicle search.

    Receives { search_id, license_plate, callback_url } from Django,
    returns HTTP 200 immediately, and runs the search in a daemon thread.
    Results are delivered to callback_url once the scan finishes.
    """
    """
    Receives from Django:
        { "search_id": int, "license_plate": str, "callback_url": str }
    Returns 200 immediately, processes in background.
    """
    try:
        data = request.get_json()

        search_id     = data.get('search_id')
        license_plate = data.get('license_plate', '').strip().upper()
        callback_url  = data.get('callback_url')

        if not search_id or not license_plate or not callback_url:
            return jsonify({
                'success': False,
                'error': 'Missing: search_id, license_plate, callback_url'
            }), 400

        print(f"\n{'='*60}")
        print(f"  New search request received")
        print(f"   search_id     : {search_id}")
        print(f"   license_plate : {license_plate}")
        print(f"   callback_url  : {callback_url}")
        print(f"{'='*60}")

        # Daemon thread â€” killed automatically when the notebook process exits.
        worker = threading.Thread(
            target=search_and_callback,
            args=(search_id, license_plate, callback_url),
            daemon=True
        )
        worker.start()

        return jsonify({
            'success': True,
            'message': f'Scan started for {license_plate}. Results will be sent to callback.'
        }), 200

    except Exception as e:
        print(f"  Error in /search endpoint: {e}")
        return jsonify({'success': False, 'error': str(e)}), 500



# Background worker: full 3-phase incident detection pipeline.
def detect_and_callback(incident_log_id, video_path, callback_url):
    """Run the complete incident pipeline in a background thread and POST results to Django.

    Phase 1 - Incident detection:
        Scans the video with the custom model. Rolling-window scoring confirms
        an accident or fight when enough positives accumulate.

    Phase 2 - Plate extraction (accidents only):
        Re-opens the video for the pre-impact window. Crops, upscales, and
        OCR-reads the rear of each car near the impact zone to identify plates.

    Phase 3 - Cross-camera tracking:
        Launches one scan_camera_feed thread per camera for each detected plate.

    Finally, POSTs the complete payload to callback_url. On failure sends a
    minimal fallback so Django is not left waiting indefinitely.
    """
    try:
        print(f"\n{'='*60}")
        print(f"  Starting detection pipeline")
        print(f"  incident_log_id : {incident_log_id}")
        print(f"  video_path      : {video_path}")
        print(f"  callback_url    : {callback_url}")
        print(f"{'='*60}")

        # ---- PHASE 1: Incident Detection ----
        print("\n PHASE 1: Incident Detection...")
        confidence_threshold = CONFIDENCE_THRESHOLD
        window_size = DETECTION_WINDOW_SIZE
        activation_threshold = ACTIVATION_THRESHOLD

        cap = cv2.VideoCapture(video_path)
        fps = get_video_fps(video_path)

        incident_snapshot = None
        incident_frame = 0
        impact_centers = []
        incident_type = None

        recent_accidents = deque(maxlen=window_size)
        recent_fights = deque(maxlen=window_size)

        frame_idx = 0
        while cap.isOpened():
            ret, frame = cap.read()
            if not ret:
                break
            frame_idx += 1

            results = custom_model.predict(frame, conf=confidence_threshold, verbose=False)
            frame_classes = [custom_model.names[int(box.cls[0].item())] for box in results[0].boxes]

            is_acc = 'accident' in frame_classes
            is_fgt = 'fight' in frame_classes

            recent_accidents.append(1 if is_acc else 0)
            recent_fights.append(1 if is_fgt else 0)

            acc_score = sum(recent_accidents)
            fgt_score = sum(recent_fights)

            if acc_score >= activation_threshold:
                incident_type = 'accident'
                incident_snapshot = frame.copy()
                incident_frame = frame_idx
                impact_centers = extract_impact_centers(frame)
                break
            elif fgt_score >= activation_threshold:
                incident_type = 'fight'
                incident_snapshot = frame.copy()
                incident_frame = frame_idx
                break

        cap.release()

        # No incident found â€” notify Django and exit early.
        if not incident_type:
            print("  No incident detected in video.")
            payload = {
                "incident_log_id": incident_log_id,
                "status": "no_incident"
            }
            requests.post(callback_url, json=payload, timeout=30)
            print("  Callback sent (no incident)")
            return

        print(f"  Incident detected: {incident_type} at frame {incident_frame}")

        # ---- Phase 2: Plate Extraction (accidents only) ----
        # Fights do not involve vehicles so we skip plate extraction for them.
        vehicles_data = []

        if incident_type == 'accident' and impact_centers:
            print("\n PHASE 2: Extracting license plates...")
            pre_impact_frames = int(fps * PRE_IMPACT_SECONDS)
            start_frame = max(0, incident_frame - pre_impact_frames)
            end_frame = incident_frame

            cap = cv2.VideoCapture(video_path)
            plate_ballot = []

            frame_idx = 0
            while cap.isOpened():
                ret, frame = cap.read()
                if not ret:
                    break
                frame_idx += 1
                if frame_idx < start_frame:
                    continue
                if frame_idx > end_frame:
                    break

                results = tracking_model.track(frame, persist=True, conf=0.4, verbose=False)
                if results[0].boxes.id is None:
                    continue

                boxes = results[0].boxes.xyxy.cpu().numpy().astype(int)
                ids = results[0].boxes.id.cpu().numpy().astype(int)
                clss = results[0].boxes.cls.cpu().numpy().astype(int)

                for box, id, cls in zip(boxes, ids, clss):
                    if results[0].names[cls] == 'car':
                        cx = (box[0] + box[2]) // 2
                        cy = (box[1] + box[3]) // 2
                        # Only process cars within the impact zone.
                        if impact_centers and not any(
                            np.linalg.norm(np.array([cx, cy]) - np.array(ic)) < IMPACT_ZONE_RADIUS
                            for ic in impact_centers
                        ):
                            continue

                        x1, y1, x2, y2 = box
                        h = y2 - y1
                        # Crop the lower bumper area where the license plate is located.
                        bumper_crop = frame[int(y1 + h * 0.50): y2 + 5, max(0, x1 - 5): min(frame.shape[1], x2 + 5)]

                        processed = forensic_upscale(bumper_crop)
                        if processed is None:
                            continue
                        if is_blurry(processed):
                            continue

                        ocr_hits = reader.readtext(processed, detail=0, allowlist='0123456789ABCDEFGHIJKLMNOPQRSTUVWXYZ')
                        for text in ocr_hits:
                            detected_clean = clean_text(text)
                            # Strip a stray leading digit OCR sometimes prepends.
                            if len(detected_clean) > 6 and detected_clean[0] in "01":
                                detected_clean = detected_clean[1:]
                            if len(detected_clean) >= 6:
                                plate_ballot.append(detected_clean)

            cap.release()

            # Select the most credible plate from all OCR readings.
            detected_plates = []
            if plate_ballot:
                valid_plates = []
                for plate in plate_ballot:
                    if len(plate) == 6:
                        letters = sum(1 for c in plate if c.isalpha())
                        digits = sum(1 for c in plate if c.isdigit())
                        if letters == 3 and digits == 3:
                            valid_plates.append(plate)

                if valid_plates:
                    # Use the most frequently seen plate as the consensus answer.
                    valid_counter = Counter(valid_plates)
                    most_common = valid_counter.most_common(1)[0]
                    detected_plates = [most_common[0]]
                else:
                    # Fallback: score by prefix frequency and character-position frequency.
                    prefixes = [plate[:3] for plate in plate_ballot if len(plate) >= 3]
                    if prefixes:
                        prefix_counter = Counter(prefixes)
                        most_common_prefix = prefix_counter.most_common(1)[0][0]
                        prefix_plates = [p for p in plate_ballot if p.startswith(most_common_prefix)]
                        unique_plates = list(set(prefix_plates))
                        scored_plates = []
                        for plate in unique_plates:
                            letter_count = sum(1 for c in plate if c.isalpha())
                            occurrence_count = prefix_plates.count(plate)
                            total_score = (letter_count * 10) + (occurrence_count * 5)
                            scored_plates.append((plate, total_score))
                        scored_plates.sort(key=lambda x: x[1], reverse=True)
                        if scored_plates:
                            detected_plates = [scored_plates[0][0]]

            print(f"  Found {len(detected_plates)} plates: {detected_plates}")

            # ---- Phase 3: Track each plate across all camera feeds ----
            for plate in detected_plates:
                print(f"\n PHASE 3: Tracking plate {plate} across cameras...")

                threads = []
                for cam_id, cam_info in CAMERA_FEEDS.items():
                    thread = threading.Thread(
                        target=scan_camera_feed,
                        args=(cam_id, cam_info, plate)
                    )
                    threads.append(thread)
                    thread.start()

                for thread in threads:
                    thread.join()

                # Collect camera sightings and encode snapshots for the payload.
                detections = []
                while not results_queue.empty():
                    result_data = results_queue.get()
                    if result_data.get('result') is not None:
                        det = result_data['result']
                        # Convert snapshot from numpy array to base64
                        if det.get('snapshot') is not None and not isinstance(det['snapshot'], str):
                            det['snapshot'] = image_to_base64(det['snapshot'])
                        detections.append(det)

                vehicles_data.append({
                    "plate": plate,
                    "confidence": 0.9,
                    "detections": detections
                })

        # Build the final payload and POST it to the Django callback URL.
        payload = {
            "incident_log_id": incident_log_id,
            "incident_type": incident_type,
            "location": "Main Camera Feed",
            "snapshot": image_to_base64(incident_snapshot) if incident_snapshot is not None else None,
            "vehicles": vehicles_data
        }

        print(f"\n Posting results back to Django: {callback_url}")
        print(f"   incident_log_id={incident_log_id}, type={incident_type}, vehicles={len(vehicles_data)}")

        response = requests.post(callback_url, json=payload, timeout=30)

        if response.status_code in (200, 201):
            print(f"  Django received results successfully (HTTP {response.status_code})")
        else:
            print(f"  Django returned HTTP {response.status_code}: {response.text}")

    except Exception as e:
        print(f"  Error in detect_and_callback: {e}")
        # Send a minimal fallback so Django can mark the log as processed.
        try:
            fallback = {
                "incident_log_id": incident_log_id,
                "status": "no_incident"
            }
            requests.post(callback_url, json=fallback, timeout=10)
            print("  Fallback callback sent.")
        except Exception as fe:
            print(f"  Fallback callback also failed: {fe}")

@app.route('/detect_incident_async', methods=['POST'])
def detect_incident_async():
    """Async endpoint that triggers the full incident detection pipeline.

    Receives { incident_log_id, video_path, callback_url } from Django,
    returns HTTP 200 immediately, and runs the 3-phase pipeline in a daemon
    background thread. Results are delivered to callback_url asynchronously.
    """
    try:
        data = request.get_json()

        incident_log_id = data.get('incident_log_id')
        video_path      = data.get('video_path')
        callback_url    = data.get('callback_url')

        if not incident_log_id or not video_path or not callback_url:
            return jsonify({
                'success': False,
                'error': 'Missing: incident_log_id, video_path, callback_url'
            }), 400

        print(f"\n{'='*60}")
        print(f"  New detection request received")
        print(f"   incident_log_id : {incident_log_id}")
        print(f"   video_path      : {video_path}")
        print(f"   callback_url    : {callback_url}")
        print(f"{'='*60}")

        # Daemon thread â€” automatically stopped when the notebook process exits.
        worker = threading.Thread(
            target=detect_and_callback,
            args=(incident_log_id, video_path, callback_url),
            daemon=True
        )
        worker.start()

        return jsonify({
            'success': True,
            'message': f'Detection started for log {incident_log_id}. Results will be sent to callback.'
        }), 200

    except Exception as e:
        print(f"  Error in /detect_incident_async endpoint: {e}")
        return jsonify({'success': False, 'error': str(e)}), 500


if __name__ == '__main__':
    app.run(host='0.0.0.0', port=5001)

 * Serving Flask app '__main__'
 * Debug mode: off


 * Running on all addresses (0.0.0.0)
 * Running on http://127.0.0.1:5001
 * Running on http://172.19.2.2:5001
Press CTRL+C to quit
127.0.0.1 - - [25/Feb/2026 07:57:22] "POST /search HTTP/1.1" 200 -



  New search request received
   search_id     : 30
   license_plate : JV316S
   callback_url  : https://nonclimbable-muddledly-jami.ngrok-free.dev/api/search/result/

  Starting search for plate: JV316S
  [cam1] Thread started - Scanning Main Street Intersection...
  [cam2] Thread started - Scanning Highway Exit 42...
  [cam3] Thread started - Scanning Downtown Plaza...
       [cam3] MATCH FOUND at 2026-02-25 07:57:33
       [cam2] Scan complete - No match found
       [cam1] Scan complete - No match found
  Posting search results to Django: https://nonclimbable-muddledly-jami.ngrok-free.dev/api/search/result/
   search_id=30, plate=JV316S, detections=1
  Django received results (HTTP 200)


127.0.0.1 - - [25/Feb/2026 08:03:11] "POST /detect_incident_async HTTP/1.1" 200 -



  New detection request received
   incident_log_id : 3
   video_path      : /kaggle/input/datasets/shiranofficial/roadsurveillance-testdata/fight1.mp4
   callback_url    : https://nonclimbable-muddledly-jami.ngrok-free.dev/api/incident/create/

  Starting detection pipeline
  incident_log_id : 3
  video_path      : /kaggle/input/datasets/shiranofficial/roadsurveillance-testdata/fight1.mp4
  callback_url    : https://nonclimbable-muddledly-jami.ngrok-free.dev/api/incident/create/

 PHASE 1: Incident Detection...
  Incident detected: fight at frame 15

 Posting results back to Django: https://nonclimbable-muddledly-jami.ngrok-free.dev/api/incident/create/
   incident_log_id=3, type=fight, vehicles=0
  Django received results successfully (HTTP 200)


127.0.0.1 - - [25/Feb/2026 08:06:14] "POST /detect_incident_async HTTP/1.1" 200 -



  New detection request received
   incident_log_id : 4
   video_path      : /kaggle/input/datasets/shiranofficial/roadsurveillance-testdata/accident.mp4
   callback_url    : https://nonclimbable-muddledly-jami.ngrok-free.dev/api/incident/create/

  Starting detection pipeline
  incident_log_id : 4
  video_path      : /kaggle/input/datasets/shiranofficial/roadsurveillance-testdata/accident.mp4
  callback_url    : https://nonclimbable-muddledly-jami.ngrok-free.dev/api/incident/create/

 PHASE 1: Incident Detection...
  Incident detected: accident at frame 346

 PHASE 2: Extracting license plates...
  Found 1 plates: ['845ZDS']

 PHASE 3: Tracking plate 845ZDS across cameras...
  [cam1] Thread started - Scanning Main Street Intersection...
  [cam2] Thread started - Scanning Highway Exit 42...
  [cam3] Thread started - Scanning Downtown Plaza...
       [cam2] MATCH FOUND at 2026-02-25 08:07:02
       [cam3] Scan complete - No match found
       [cam1] Scan complete - No match found

 Pos

127.0.0.1 - - [25/Feb/2026 08:09:40] "POST /search HTTP/1.1" 200 -



  New search request received
   search_id     : 31
   license_plate : X985VD
   callback_url  : https://nonclimbable-muddledly-jami.ngrok-free.dev/api/search/result/

  Starting search for plate: X985VD
  [cam1] Thread started - Scanning Main Street Intersection...
  [cam2] Thread started - Scanning Highway Exit 42...
  [cam3] Thread started - Scanning Downtown Plaza...
       [cam1] MATCH FOUND at 2026-02-25 08:10:40
       [cam3] Scan complete - No match found
       [cam2] Scan complete - No match found
  Posting search results to Django: https://nonclimbable-muddledly-jami.ngrok-free.dev/api/search/result/
   search_id=31, plate=X985VD, detections=1
  Django received results (HTTP 200)
